# Notebook_C (shared, data section, Student A): collection, source comparison, cleaning, EDA, and SQL

**Shared across every topic in the course.** Just fill in the CONFIG block below (topic name, sources, column names, frequency) and run sequentially. Every code cell has English comments; explanations, reasoning, and how to read results are in English right above each cell.

**Deliverables handed off to Student B**: `data/processed/feat.parquet`, `data/processed/manifest.json`, the `report/` folder (RQ1 tables, figures, cleaning log, AI Audit Log).

**Three shared research questions (apply to every topic; each group adds 1-2 of their own):**

* **RQ1 (data and description, in depth)**: Are the two data sources (primary and secondary) consistent with each other, how does the target variable vary by season, by unit, and by covariate, and how much of the variation is explained by the calendar versus the secondary source?
* **RQ2 (modeling and generalization)**: Among six model families (naive baseline, ridge, random forest, global LightGBM, long-horizon linear, zero-shot foundation model), which performs best by forecast horizon, and do results hold up under a time-based train/test split and on unseen units?
* **RQ3 (reliability and operations)**: Do confidence intervals achieve correct coverage, what recall and precision does event warning (target exceeding a high quantile) achieve, how does error change across regimes, and how much does the secondary source contribute (ablation)?

Notebook_C answers RQ1 and prepares the data for RQ2, RQ3 (Notebook_D).

In [2]:
import pandas as pd

m = pd.read_parquet('data/processed/solar.parquet')

# Primary: plant power output only
primary = m[['plant', 'ts', 'power', 'cap_mw', 'cf']].drop_duplicates()
primary.to_csv('data/raw/primary.csv', index=False)

# Secondary: weather data only (drop duplicate plant-ts weather rows if plants share sites)
secondary = m[['plant', 'ts', 'GHI', 'DNI', 'cloud', 'temp']].drop_duplicates()
secondary.to_csv('data/raw/secondary.csv', index=False)

print(primary.shape, secondary.shape)

(87302, 5) (87302, 6)


In [3]:
# CONFIG: fill in once for your group, both notebooks read the same block
TOPIC = "Cross-plant Transfer and Conformal Uncertainty for Hourly Solar Power Forecasting on NREL Integration Data"
GROUP = "Group 7"                    # group label used in file names and the audit log
PRIMARY_SOURCE = {"name": "NREL/NLR Solar Power Data for Integration Studies (California)", "url": "https://www.nrel.gov/grid/solar-power-data", "license": "Public access (specific license unconfirmed)", "path": "data/raw/primary.csv"}
SECOND_SOURCE  = {"name": "NSRDB (National Solar Radiation Database) weather data", "url": "https://nsrdb.nlr.gov", "license": "CC BY 4.0", "path": "data/raw/secondary.csv"}
UNIT_COL   = "plant"                  # column that identifies a series (station, site, zone, vm, ...)
TIME_COL   = "ts"                    # timestamp column (will be parsed to datetime)
TARGET_COL = "cf"                     # variable to forecast
EXOG_COLS  = ["GHI", "DNI", "cloud", "temp"]      # columns from the secondary source used as covariates
FREQ       = "h"                     # pandas offset alias of the regular grid: 'h', 'D', 'W', '15min', 'min'
HORIZONS   = [1, 6]                 # forecast horizons in steps of FREQ (e.g. 1 h and 24 h ahead)
SEASON     = 24                      # seasonal period in steps (24 for hourly-daily, 7 for daily-weekly, 52 for weekly-yearly)
TEST_START = "2006-10-01"            # first timestamp of the test period (time-based split)
EVENT_QUANTILE = 0.9                 # "event" = target above this quantile (used by RQ3 warning metrics)
print("Topic:", TOPIC); print("Group:", GROUP)

Topic: Cross-plant Transfer and Conformal Uncertainty for Hourly Solar Power Forecasting on NREL Integration Data
Group: Group 7


In [4]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report", exist_ok=True)
AUDIT_PATH = "report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

## Guide to asking AI (fill in your group's topic where blank)

Rule: ask **specifically** (include column names, sizes, error messages), require the AI to **explain its reasoning** and **state how to verify it**, then check it yourself before using it. Every prompt that changes your work must be logged in the AI Audit Log using the `audit(...)` function in the cell above. Prompt templates by step (replace `{TOPIC}`, `{column}` with your group's real information):

| Step | Prompt template | What to verify yourself |
|---|---|---|
| Collection | "My topic is {TOPIC}. The primary source is {PRIMARY_SOURCE}. Suggest 3 public secondary sources (weather, events, prices, calendar) that can be joined on {TIME_COL} and {UNIT_COL}, with download URLs and licenses." | Open the URL, confirm it downloads and the license permits use |
| Source comparison | "I have two sources for the same variable {TARGET_COL} at frequency {FREQ}. Suggest 4 metrics to compare coverage, update lag, and bias between the two sources." | Recompute the numbers on real data, don't use the AI's numbers directly |
| Cleaning | "Column {column} is {x}% missing, with the longest missing streak {n} steps. Should I interpolate or drop? Explain the risk of future leakage." | Check that interpolation doesn't use future values |
| SQL | "Write a DuckDB query that creates a {SEASON}-step lag feature and rolling mean by {UNIT_COL}, using WINDOW, with no leakage." | Count rows, check for NULL columns at the start of each series |
| EDA | "From this summary table (paste table), give 3 verifiable observations and 2 things to be cautious about." | Every observation must point to a specific number in the table |
| Errors | "Error: {paste verbatim}. Context: {code cell}. Cause and minimal fix?" | Rerun the test cell after fixing |

Example Audit Log entry after asking: `audit("Collection", "Topic ... suggest secondary sources", "Claude", "3 sources: NOAA ISD, ...", "opened 3 URLs, 1 URL returned 404", "used NOAA ISD, excluded the 404 source", hallucination=True)`.

## Step 0: environment and folder structure

**What to do**: create a virtual environment once, install libraries, create the standard folders. **Why**: if both students and the instructor run the same library versions, results are reproducible; `requirements.txt` is the evidence.

**Structure**: `data/raw` (original files, never edited), `data/processed` (clean parquet), `sql/` (queries), `report/` (tables, figures, logs), `notebooks/`.

In [5]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests
# pip freeze > requirements.txt
import os
for d in ["data/raw", "data/processed", "sql", "report", "notebooks"]: os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [6]:
# Step 0b: imports and plotting style (English labels in figures, no top/right spines, no in-figure titles)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


## Step 1: the problem, data sources, and collection plan

**What to do**: write the problem in 5 sentences, register both sources (primary and secondary) in the source registry (`report/data_sources.csv`) with URL, license, download date, row count. **Why**: the paper's Data section needs to state clearly which work the source comes from, which year, how many rows and columns, and why it was chosen; the source registry lets you write that section in two minutes.

**Choosing a secondary source**: pick a variable with a clear mechanism affecting the target (weather with demand, price with charging, events with traffic), at the same or higher frequency than the target, and joinable by time and unit.

In [7]:
# Step 1a: source registry (append every file you download; this becomes Table "Data sources" in the paper)
SOURCES_PATH = "report/data_sources.csv"
def register_source(src, rows=None, cols=None, start=None, end=None, note=""):
    """src: dict with name, url, license, path. rows/cols/start/end are filled after loading."""
    row = {**src, "downloaded": pd.Timestamp.today().date().isoformat(), "rows": rows, "cols": cols, "start": start, "end": end, "note": note}
    df = pd.DataFrame([row]); df.to_csv(SOURCES_PATH, mode="a", header=not os.path.exists(SOURCES_PATH), index=False)
    return row
print("registry at", SOURCES_PATH)

registry at report/data_sources.csv


In [8]:
# Step 1b: load the primary source (edit the reader to match your file: csv, parquet, or an API export saved to data/raw)
def load_table(path):
    """Read csv/parquet with DuckDB so large files never need to fit in memory; returns a pandas DataFrame."""
    if path.endswith(".parquet"): return con.execute(f"SELECT * FROM read_parquet('{path}')").df()
    return con.execute(f"SELECT * FROM read_csv_auto('{path}', union_by_name=true, sample_size=-1)").df()
raw = load_table(PRIMARY_SOURCE["path"])
raw.columns = [c.strip() for c in raw.columns]
raw[TIME_COL] = pd.to_datetime(raw[TIME_COL], errors="coerce", utc=False)
print(raw.shape); display(raw.head(3)); display(raw.dtypes.value_counts())
register_source(PRIMARY_SOURCE, rows=len(raw), cols=raw.shape[1], start=str(raw[TIME_COL].min()), end=str(raw[TIME_COL].max()))

(87302, 5)


,plant,ts,power,cap_mw,cf
0,Actual_33.65_-117.25_2006_DPV_,2006-01-01 07:00:00,0.066667,26.916667,0.002477
1,Actual_33.65_-117.25_2006_DPV_,2006-01-01 08:00:00,5.266667,26.916667,0.195666
2,Actual_33.65_-117.25_2006_DPV_,2006-01-01 09:00:00,8.341667,26.916667,0.309907


float64           3
object            1
datetime64[us]    1
Name: count, dtype: int64

{'name': 'NREL/NLR Solar Power Data for Integration Studies (California)',
 'url': 'https://www.nrel.gov/grid/solar-power-data',
 'license': 'Public access (specific license unconfirmed)',
 'path': 'data/raw/primary.csv',
 'downloaded': '2026-09-18',
 'rows': 87302,
 'cols': 5,
 'start': '2006-01-01 07:00:00',
 'end': '2006-12-31 17:00:00',
 'note': ''}

In [9]:
raw = raw.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

In [10]:
# No second source, everything is inside 1 single file from notebook A
# Step 1c: load the secondary source (covariates); it must share TIME_COL (and UNIT_COL if it is unit-specific)
sec = load_table(SECOND_SOURCE["path"]); sec.columns = [c.strip() for c in sec.columns]
sec[TIME_COL] = pd.to_datetime(sec[TIME_COL], errors="coerce")
SEC_HAS_UNIT = UNIT_COL in sec.columns
print(sec.shape, "unit-specific:", SEC_HAS_UNIT); display(sec.head(3))
register_source(SECOND_SOURCE, rows=len(sec), cols=sec.shape[1], start=str(sec[TIME_COL].min()), end=str(sec[TIME_COL].max()))

(87302, 6) unit-specific: True


,plant,ts,GHI,DNI,cloud,temp
0,Actual_33.65_-117.25_2006_DPV_,2006-01-01 07:00:00,75,433,0,10.6
1,Actual_33.65_-117.25_2006_DPV_,2006-01-01 08:00:00,163,99,7,11.8
2,Actual_33.65_-117.25_2006_DPV_,2006-01-01 09:00:00,155,14,8,12.9


{'name': 'NSRDB (National Solar Radiation Database) weather data',
 'url': 'https://nsrdb.nlr.gov',
 'license': 'CC BY 4.0',
 'path': 'data/raw/secondary.csv',
 'downloaded': '2026-09-18',
 'rows': 87302,
 'cols': 6,
 'start': '2006-01-01 07:00:00',
 'end': '2006-12-31 17:00:00',
 'note': ''}

### Exporting and reading data back as CSV

**File convention**: every step also saves a separate CSV so the instructor and teammates can open it in Excel without Python: `data/raw/primary_snapshot.csv` (original with standardized column names and time type, values unmodified), `data/processed/clean.csv` (after cleaning, one row per unit-timestamp), `data/processed/feat.csv` (the handover feature table). Parquet is still saved in parallel since it's fast and preserves exact data types; CSV is the version to read and submit. When reading a CSV back, always coerce the time type with `parse_dates` and check the row count matches the manifest.

In [11]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "data/raw/primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

wrote data/raw/primary_snapshot.csv | rows 87302 | columns ['plant', 'ts', 'power', 'cap_mw', 'cf']


### Step 1d: collecting more data (extending the time range or units)

**What to do**: if the source allows it, download more periods or more units (stations, zones) so the table reaches roughly 100 thousand rows and covers at least two seasons or two regimes. The function below combines multiple downloaded files by year or by unit and removes duplicates. **Why**: a global model and testing on unseen units both need many units; a regime change can only be measured with data from before and after it.

In [12]:
# Step 1d: combine several downloaded chunks (data/raw/primary_*.csv) into one table, deduplicated on unit and time
import glob
chunks = sorted(glob.glob(PRIMARY_SOURCE["path"].replace(".csv", "_*.csv")))
if chunks:
    extra = pd.concat([load_table(f) for f in chunks]); extra[TIME_COL] = pd.to_datetime(extra[TIME_COL], errors="coerce")
    before = len(raw); raw = pd.concat([raw, extra]).drop_duplicates([UNIT_COL, TIME_COL] if UNIT_COL in raw.columns else [TIME_COL])
    print(f"added {len(raw) - before} rows from {len(chunks)} chunks")
else:
    print("no extra chunks found (pattern primary_*.csv); skip if your source is a single file")
if UNIT_COL not in raw.columns: raw[UNIT_COL] = "all"; print("single-series data: UNIT_COL set to 'all'")

added 0 rows from 1 chunks


## Step 2: compare the two sources and clean with a log

**Source comparison (RQ1a)**: before merging, measure (1) the overlapping time range, (2) the share of time steps present in one source but not the other, (3) if both sources share a common variable, their correlation and mean bias, (4) update lag (the latest timestamp of each source). This table goes into the paper's Data section.

**Cleaning with a log**: every operation records rows before, after, and the reason; the log table is the evidence that convinces a reader to trust the data.

In [13]:
# Step 2a: source comparison table
def compare_sources(a, b, time_col, common_var=None):
    ta, tb = a[time_col].dropna(), b[time_col].dropna()
    ov_start, ov_end = max(ta.min(), tb.min()), min(ta.max(), tb.max())
    ga = set(ta.dt.floor(FREQ).unique()); gb = set(tb.dt.floor(FREQ).unique())
    rows = [["overlap start", ov_start], ["overlap end", ov_end], ["steps only in primary", len(ga - gb)], ["steps only in secondary", len(gb - ga)],
            ["latest timestamp primary", ta.max()], ["latest timestamp secondary", tb.max()]]
    if common_var and common_var in a.columns and common_var in b.columns:
        m = a.groupby(time_col)[common_var].mean().to_frame("a").join(b.groupby(time_col)[common_var].mean().to_frame("b"), how="inner")
        rows += [["correlation of common variable", round(m.a.corr(m.b), 3)], ["mean difference (primary minus secondary)", round((m.a - m.b).mean(), 3)]]
    return pd.DataFrame(rows, columns=["metric", "value"])
cmp_tbl = compare_sources(raw, sec, TIME_COL, common_var=None)     # set common_var="temp" if both sources carry the same variable
cmp_tbl.to_csv("report/table_source_comparison.csv", index=False); display(cmp_tbl)

,metric,value
0,overlap start,2006-01-01 07:00:00
1,overlap end,2006-12-31 17:00:00
2,steps only in primary,0
3,steps only in secondary,0
4,latest timestamp primary,2006-12-31 17:00:00
5,latest timestamp secondary,2006-12-31 17:00:00


Both sources are perfectly consistent

In [14]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason
clean_log = []
def step(df, name, fn, reason):
    n0 = len(df); out = fn(df); clean_log.append([name, n0, len(out), n0 - len(out), reason]); return out
df = raw.copy()
df = step(df, "parse time", lambda x: x.dropna(subset=[TIME_COL]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([UNIT_COL, TIME_COL]), "same unit and timestamp")
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[TARGET_COL]), "target missing")
lo, hi = df[TARGET_COL].quantile([0.001, 0.999])
df = step(df, "physical range", lambda x: x[(x[TARGET_COL] >= lo) & (x[TARGET_COL] <= hi)], f"outside [{lo:.3g}, {hi:.3g}] (0.1% tails; replace with a physical range if you know it)")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
display(pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]))

,step,rows_before,rows_after,rows_removed,reason
0,parse time,87302,87302,0,rows with unparseable timestamps
1,drop duplicates,87302,87302,0,same unit and timestamp
2,drop missing target,87302,87302,0,target missing
3,physical range,87302,87214,88,"outside [0, 0.968] (0.1% tails; replace with a..."


In [15]:
# Step 2c: regular time grid per unit, then short-gap interpolation (limit = 3 steps, never across long gaps)
df = df.set_index(TIME_COL).groupby(UNIT_COL)[[TARGET_COL] + [c for c in df.columns if c not in (UNIT_COL, TIME_COL, TARGET_COL) and pd.api.types.is_numeric_dtype(df[c])]].resample(FREQ).mean().reset_index()
gap = df.groupby(UNIT_COL)[TARGET_COL].apply(lambda s: s.isna().mean()).rename("missing_share")
df[TARGET_COL] = df.groupby(UNIT_COL)[TARGET_COL].transform(lambda s: s.interpolate(limit=3))
n_before = len(df); df = df.dropna(subset=[TARGET_COL]); clean_log.append(["regular grid + interpolate(limit=3)", n_before, len(df), n_before - len(df), "gaps longer than 3 steps dropped"])
print("missing share per unit before interpolation:"); display(gap.describe().round(3))

missing share per unit before interpolation:


count    20.000
mean      0.501
std       0.002
min       0.500
25%       0.500
50%       0.500
75%       0.502
max       0.506
Name: missing_share, dtype: float64

In [16]:
# Step 2d: keep units with enough history (at least 90% of the longest unit) so the global model is not dominated by fragments
cnt = df.groupby(UNIT_COL).size(); keep = cnt[cnt >= 0.9 * cnt.max()].index
n_before = len(df); df = df[df[UNIT_COL].isin(keep)]; clean_log.append(["drop short units", n_before, len(df), n_before - len(df), f"{len(cnt) - len(keep)} units with < 90% coverage"])
print("units kept:", len(keep), "of", len(cnt))

units kept: 20 of 20


In [17]:
# Step 2e: merge the secondary source (nearest timestamp within one step; by unit if the secondary source is unit-specific)
sec_num = sec[[TIME_COL] + ([UNIT_COL] if SEC_HAS_UNIT else []) + [c for c in EXOG_COLS if c in sec.columns]].copy()
for c in EXOG_COLS:
    if c in sec_num.columns: sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")
sec_num = sec_num.sort_values(TIME_COL); df = df.sort_values(TIME_COL)
tol = pd.Timedelta(pd.tseries.frequencies.to_offset(FREQ))
m = pd.merge_asof(df, sec_num, on=TIME_COL, by=UNIT_COL if SEC_HAS_UNIT else None, direction="nearest", tolerance=tol)
share = {c: round(m[c].notna().mean(), 3) for c in EXOG_COLS if c in m.columns}
print("share of rows with each covariate:", share); df = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

share of rows with each covariate: {'GHI': np.float64(0.867), 'DNI': np.float64(0.867), 'cloud': np.float64(0.867), 'temp': np.float64(0.867)}


In [18]:
# Step 2f: save the clean table and the cleaning log; print a one-paragraph data statement for the paper
df.to_parquet("data/processed/clean.parquet", index=False)
df.to_csv("data/processed/clean.csv", index=False)                     # separate clean CSV for submission and for opening in Excel
chk = pd.read_csv("data/processed/clean.csv", parse_dates=[TIME_COL]); assert len(chk) == len(df) and chk[TARGET_COL].notna().all(), "clean.csv round trip failed"
print("wrote data/processed/clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
print(f"Data statement: {PRIMARY_SOURCE['name']} ({PRIMARY_SOURCE['license']}) merged with {SECOND_SOURCE['name']}; {len(df):,} rows, {df[UNIT_COL].nunique()} units, "
      f"{df[TIME_COL].min().date()} to {df[TIME_COL].max().date()} at frequency {FREQ}; {sum(r[3] for r in clean_log):,} rows removed by cleaning.")

wrote data/processed/clean.csv with 109141 rows
Data statement: NREL/NLR Solar Power Data for Integration Studies (California) (Public access (specific license unconfirmed)) merged with NSRDB (National Solar Radiation Database) weather data; 109,141 rows, 20 units, 2006-01-01 to 2006-12-31 at frequency h; 65,865 rows removed by cleaning.


## EDA (RQ1): description, seasonality, units, secondary source, autocorrelation

**How to read it**: each figure answers one small question and records one verifiable observation in its caption. Don't plot a figure just to make it look nice. Order: (1) overall distribution, (2) time series for a few units, (3) calendar seasonality, (4) differences between units, (5) relationship with the secondary source and lag, (6) autocorrelation deciding lag features, (7) missing data.

In [19]:
# EDA 1: automatic profile (open report/profile.html in a browser) and summary statistics
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, title=f"Profile {GROUP}", minimal=True).to_file("report/profile.html")
desc = df[[TARGET_COL] + [c for c in EXOG_COLS if c in df.columns]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3)
desc.to_csv("report/table_describe.csv"); display(desc)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:00<00:00, 529.24it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

,count,mean,std,min,1%,5%,50%,95%,99%,max
cf,109141.0,0.363,0.297,0.0,0.0,0.001,0.337,0.813,0.889,0.968
GHI,94581.0,434.205,312.655,2.0,9.0,22.000,412.000,967.000,1028.000,1078.000
DNI,94581.0,560.110,350.806,0.0,0.0,0.000,671.000,962.000,987.000,1032.000
cloud,94581.0,1.753,2.772,0.0,0.0,0.000,0.000,8.000,8.000,10.000
temp,94581.0,22.428,9.836,-2.8,4.3,7.700,21.700,39.200,43.500,49.700


In [20]:
# EDA 2: distribution of the target (histogram with many bins) and log-scale check for skewed data
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(df[TARGET_COL], bins=60, color=TEAL); axes[0].set_xlabel(TARGET_COL); axes[0].set_ylabel("Count"); style(axes[0])
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC); axes[1].set_xlabel("log1p(" + TARGET_COL + ")"); style(axes[1])
savefig(fig, "fig_eda_distribution"); print("skewness:", round(df[TARGET_COL].skew(), 2), "-> consider modelling log1p if skewness > 2")

saved report/fig_eda_distribution.png
skewness: 0.18 -> consider modelling log1p if skewness > 2


In [21]:
print((df[TARGET_COL] == 0).sum(), "rows with cf == 0")
print((df[TARGET_COL] == 0).mean().round(3), "fraction of rows with cf == 0")
print(df[df[TARGET_COL]==0][['GHI']].describe())

3077 rows with cf == 0
0.028 fraction of rows with cf == 0
               GHI
count  2813.000000
mean     49.142197
std      57.640271
min       2.000000
25%      14.000000
50%      27.000000
75%      61.000000
max     395.000000


In [22]:
print(df[df[TARGET_COL]==0]['GHI'].isna().sum())

264


In [23]:
# EDA 3: time series of the first four units over the whole period (look for trends, regime changes, gaps)
units = df[UNIT_COL].unique()[:4]
fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(units, [TEAL, ACC, GREY, "#264653"]):
    g = df[df[UNIT_COL] == u]; ax.plot(g[TIME_COL], g[TARGET_COL], lw=0.7, color=col, label=str(u))
ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--", lw=1); ax.set_ylabel(TARGET_COL); ax.legend(frameon=False, ncol=4); style(ax)
savefig(fig, "fig_eda_timeseries")

saved report/fig_eda_timeseries.png


In [24]:
# EDA 4: calendar seasonality: mean target by hour of day, day of week and month (only the ones that make sense at your FREQ)
t = df[TIME_COL]; cal = pd.DataFrame({"hour": t.dt.hour, "dow": t.dt.dayofweek, "month": t.dt.month, TARGET_COL: df[TARGET_COL]})
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, k in zip(axes, ["hour", "dow", "month"]):
    s = cal.groupby(k)[TARGET_COL].mean(); ax.plot(s.index, s.values, marker="o", color=TEAL); ax.set_xlabel(k); style(ax)
axes[0].set_ylabel("mean " + TARGET_COL); savefig(fig, "fig_eda_seasonality")
season_strength = {k: round(cal.groupby(k)[TARGET_COL].mean().std() / cal[TARGET_COL].std(), 3) for k in ["hour", "dow", "month"]}
print("seasonal strength (std of group means / total std):", season_strength)

saved report/fig_eda_seasonality.png
seasonal strength (std of group means / total std): {'hour': np.float64(0.89), 'dow': np.float64(0.032), 'month': np.float64(0.105)}


In [25]:
# EDA 5: heterogeneity across units: mean and variability per unit, sorted (global models must handle this spread)
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("report/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL, yerr=None)
ax.set_xlabel("units sorted by mean"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_units")
print("ratio of largest to smallest unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))

saved report/fig_eda_units.png
ratio of largest to smallest unit mean: 1.62


In [26]:
# EDA 6: relation with covariates: binned means and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, p = spearmanr(ok[c], ok[TARGET_COL]); rows.append([c, round(rho, 3), p, len(ok)])
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"))[TARGET_COL].mean()
    ax.plot(range(len(b)), b.values, marker="o", color=TEAL); ax.set_xlabel(c + " (deciles)"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_covariate_" + c)
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"]); corr_tbl.to_csv("report/table_covariate_corr.csv", index=False); display(corr_tbl)

saved report/fig_eda_covariate_GHI.png
saved report/fig_eda_covariate_DNI.png
saved report/fig_eda_covariate_cloud.png
saved report/fig_eda_covariate_temp.png


,covariate,spearman_rho,p_value,n
0,GHI,0.867,0.0,94581
1,DNI,0.739,0.0,94581
2,cloud,-0.282,0.0,94581
3,temp,0.351,0.0,94581


In [27]:
# EDA 7: cross-correlation with lags: does the covariate lead the target? (positive lag = covariate earlier)
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 4 * SEASON + 1, max(1, SEASON // 4)))
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in [c for c in EXOG_COLS if c in df.columns][:3]:
    cc = np.nanmean([cross_corr(g, c, TARGET_COL, lags) for _, g in df.groupby(UNIT_COL) if len(g) > 5 * SEASON], axis=0)
    ax.plot(lags, cc, marker="o", label=c); print(c, "best lag:", lags[int(np.nanargmax(np.abs(cc)))], "corr:", round(float(np.nanmax(np.abs(cc))), 3))
ax.axhline(0, color=GREY); ax.set_xlabel("lag (steps, covariate leads target)"); ax.set_ylabel("correlation"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_eda_crosscorr")

GHI best lag: 0 corr: 0.87
DNI best lag: 0 corr: 0.71
cloud best lag: 0 corr: 0.243
saved report/fig_eda_crosscorr.png


In [28]:
# EDA 8: autocorrelation of the target (ACF) averaged over units: which lags to use as features and whether SEASON is right
from statsmodels.tsa.stattools import acf
nl = 2 * SEASON + 1
acfs = [acf(g[TARGET_COL].values, nlags=nl, fft=True) for _, g in df.groupby(UNIT_COL) if len(g) > 4 * SEASON]
acf_mean = np.mean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.stem(range(nl + 1), acf_mean, basefmt=" "); ax.set_xlabel("lag (steps)"); ax.set_ylabel("ACF"); style(ax); savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(acf_mean[1], 3), "| at SEASON:", round(acf_mean[SEASON], 3), "-> a seasonal naive baseline is strong if ACF at SEASON > 0.7")

saved report/fig_eda_acf.png
ACF at lag 1: 0.881 | at SEASON: -0.39 -> a seasonal naive baseline is strong if ACF at SEASON > 0.7


In [29]:
print(df[df[UNIT_COL]==df[UNIT_COL].iloc[0]][TIME_COL].dt.hour.unique())
print(df.groupby(df[TIME_COL].dt.date).size().describe())

[ 8  9 10 11 12 13 14 15 16 17 18 19 20  7 21  6 22]
count    365.000000
mean     299.016438
std       32.140826
min      198.000000
25%      266.000000
50%      300.000000
75%      335.000000
max      342.000000
dtype: float64


In [30]:
# EDA 9: missingness pattern over time per unit (heatmap of monthly missing share) and regime check (rolling mean over time)
miss = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("MS").apply(lambda s: s.isna().mean()).unstack(0)
fig, ax = plt.subplots(figsize=(10, 3.4)); sns.heatmap(miss.T, cmap="Blues", cbar_kws={"label": "missing share"}, ax=ax); ax.set_xlabel("month"); ax.set_ylabel(UNIT_COL); savefig(fig, "fig_eda_missing")
roll = df.set_index(TIME_COL)[TARGET_COL].resample("MS").mean()
fig, ax = plt.subplots(figsize=(10, 3.0)); ax.plot(roll.index, roll.values, color=TEAL, marker="o", ms=3); ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--"); ax.set_ylabel("monthly mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_regime")
print("largest month-to-month change (share of mean):", round((roll.diff().abs().max() / roll.mean()), 3), "-> above 0.5 suggests a regime shift worth reporting in RQ3")

saved report/fig_eda_missing.png
saved report/fig_eda_regime.png
largest month-to-month change (share of mean): 0.154 -> above 0.5 suggests a regime shift worth reporting in RQ3


### Recording EDA observations (required, used for the RQ1 Results section)

Fill in 5 observations, each one pointing to exactly one number in a table or figure above, e.g.: "Seasonal strength by hour is 0.42, while by month it's only 0.08, so the hour feature matters more than month (Figure EDA 4)." Don't write an observation without a number.

In [31]:
# EDA 10: write your five evidence-backed observations here (they go straight into the paper)
eda_notes = [
    "1. In fig_eda_distribution.png (Figure EDA 2), there are roughly 17500 rows that have cf = 0, however, when we checked the actual count of rows that have cf = 0, there are only 3077 rows. Because of this, it is likely the data was skewered because the figure is a histogram with bins, 60 bins across 0-1, each bin covers around 0.0167, therefore the first bin actually means a cf range from 0 to 0.0167, justifying 17500 rows.",
    "2. In fig_eda_seasonality.png (Figure EDA 4), the dow graph has a starting cf of 0.35, and the maximum at 0.375, therefore this graph is likely to be noises (seasonal strength of dow being only 0.032) as the sun does not change its behaviors on different days of the week. It can also be inferred that seasonal strength by the hour matters the most at 0.89",
    "3. In fig_eda_covariate_***.png (Figure EDA 5), all 4 covariates matter, with GHI and DNI mattering the most with a linear relationship as the decile increases, with the range of mean cf being 0.7 and 0.5 respectively. Their spearman_rho values are 0.867 and 0.739, showing strong positive correlation to cf",
    "4. In fig_eda_crosscorr.png (Figure EDA 7), the strongest correlation all happened at lag = 0, at a noticeable rhythm showing weakened effects happen roughly every 24 hours later (at the same time of day)",
    "5. In fig_eda_acf.png (Figure EDA 8), lag at 1 is high (~0.89) means cf is likely to be similar at consecutive hours, after that there's a pattern that happens every 15~16 hours, which didn't make sense until the hours are checked in the next code cell, which revealed that there are only 15 daylight hours, therefore the every 15~16 hours lag pattern makes sense as it is the same hour but in different days",
]
open("report/eda_notes.md", "w").write("\n".join(eda_notes)); print("saved report/eda_notes.md")

saved report/eda_notes.md


## Step 3: SQL with DuckDB, answering RQ1 and building the feature table

**Why SQL**: a reader can verify each number with one query; lag and rolling-window features built with `WINDOW` don't leak future information as long as they only use `LAG` and `PRECEDING`.

**Anti-leakage rule**: features may only use information **at or before** time t; the target is `LEAD(target, h)`; never use a statistic computed over the whole series (including a per-unit mean) unless it's computed only on the training portion.

In [32]:
# Step 3a: register the clean table and write the RQ1 queries to sql/queries.sql (edit the four queries to your topic; keep them minimal and readable)
con.register("clean", df)
EXOG_SQL = ", ".join([c for c in EXOG_COLS if c in df.columns]) or "NULL AS no_exog"
SQL_RQ1 = f"""
-- Q1: mean and spread of the target per unit and per calendar period
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS mean_target, STDDEV({TARGET_COL}) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the {EVENT_QUANTILE:.0%} quantile) per unit
WITH thr AS (SELECT quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS q FROM clean)
SELECT {UNIT_COL}, AVG(({TARGET_COL} > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG({TARGET_COL}) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY {EXOG_COLS[0] if EXOG_COLS else TARGET_COL}) AS decile, {TARGET_COL} FROM clean WHERE {EXOG_COLS[0] if EXOG_COLS else TARGET_COL} IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON steps earlier
SELECT corr({TARGET_COL}, lagged) AS r_season FROM (SELECT {TARGET_COL}, LAG({TARGET_COL}, {SEASON}) OVER (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL}) AS lagged FROM clean);
"""
open("sql/queries.sql", "w").write(SQL_RQ1); print(SQL_RQ1)


-- Q1: mean and spread of the target per unit and per calendar period
SELECT plant, month(ts) AS mon, AVG(cf) AS mean_target, STDDEV(cf) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the 90% quantile) per unit
WITH thr AS (SELECT quantile_cont(cf, 0.9) AS q FROM clean)
SELECT plant, AVG((cf > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG(cf) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY GHI) AS decile, cf FROM clean WHERE GHI IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON steps earlier
SELECT corr(cf, lagged) AS r_season FROM (SELECT cf, LAG(cf, 24) OVER (PARTITION BY plant ORDER BY ts) AS lagged FROM clean);



In [33]:
# Step 3b: run every statement, save each result as report/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read(); body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt: continue
        k += 1; out = con.execute(stmt).df(); out.to_csv(f"report/table_q{k}.csv", index=False); print(f"Q{k}: {out.shape}"); display(out.head(8))
run_sql_file("sql/queries.sql")

Q1: (240, 5)


,plant,mon,mean_target,sd_target,n
0,Actual_32.75_-115.45_2006_UPV_,1,0.426823,0.308524,403
1,Actual_32.75_-115.45_2006_UPV_,2,0.428650,0.311618,378
2,Actual_32.75_-115.45_2006_UPV_,3,0.452355,0.339021,461
3,Actual_32.75_-115.45_2006_UPV_,4,0.496407,0.321818,477
4,Actual_32.75_-115.45_2006_UPV_,5,0.501739,0.300868,504
5,Actual_32.75_-115.45_2006_UPV_,6,0.426066,0.276032,510
6,Actual_32.75_-115.45_2006_UPV_,7,0.424168,0.286762,527
7,Actual_32.75_-115.45_2006_UPV_,8,0.456353,0.274710,497


Q2: (20, 2)


,plant,event_share
0,Actual_37.75_-122.05_2006_UPV_,0.166240
1,Actual_33.75_-116.15_2006_UPV_,0.165720
2,Actual_33.05_-116.85_2006_DPV_,0.143955
3,Actual_33.25_-114.85_2006_UPV_,0.136506
4,Actual_32.75_-115.45_2006_UPV_,0.133579
5,Actual_34.95_-118.25_2006_UPV_,0.125800
6,Actual_33.75_-116.25_2006_DPV_,0.121540
7,Actual_37.45_-121.95_2006_DPV_,0.117228


Q3: (10, 3)


,decile,mean_target,n
0,1,0.035107,9459
1,2,0.078451,9458
2,3,0.160855,9458
3,4,0.266548,9458
4,5,0.387438,9458
5,6,0.488839,9458
6,7,0.587811,9458
7,8,0.639299,9458


Q4: (1, 1)


,r_season
0,-0.340822


In [34]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across units, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across units", round(H, 2) if H == H else "n/a", p]]
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, pv = spearmanr(ok[c], ok[TARGET_COL]); tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"]); tests.to_csv("report/table_rq1_tests.csv", index=False); display(tests)

,test,statistic,p_value
0,Kruskal-Wallis target across units,4705.950,0.0
1,Spearman target vs GHI,0.867,0.0
2,Spearman target vs DNI,0.739,0.0
3,Spearman target vs cloud,-0.282,0.0
4,Spearman target vs temp,0.351,0.0


In [35]:
# Step 3d: build the feature table with SQL (lags, rolling means, calendar, covariates, targets for every horizon); no future information
lag_list = sorted(set([1, 2, 3, SEASON, 2 * SEASON, 7 * SEASON if FREQ in ("h", "H") else SEASON * 4]))
lag_sql = ", ".join([f"LAG({TARGET_COL}, {k}) OVER w AS y_lag{k}" for k in lag_list])
roll_sql = f"AVG({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_ma_season, STDDEV({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_sd_season, " \
           f"{TARGET_COL} - LAG({TARGET_COL}, 1) OVER w AS y_diff1"
exog_lag = ", ".join([f"LAG({c}, {SEASON}) OVER w AS {c}_lag_season" for c in EXOG_COLS if c in df.columns])
targets = ", ".join([f"LEAD({TARGET_COL}, {h}) OVER w AS y_h{h}" for h in HORIZONS])
cal_sql = f"hour({TIME_COL}) AS hr, dayofweek({TIME_COL}) AS dow, month({TIME_COL}) AS mon, dayofyear({TIME_COL}) AS doy"
FEAT_SQL = f"""
CREATE OR REPLACE TABLE feat AS
SELECT {UNIT_COL}, {TIME_COL}, {TARGET_COL}, {EXOG_SQL}, {cal_sql}, {lag_sql}, {roll_sql}{', ' + exog_lag if exog_lag else ''}, {targets}
FROM clean WINDOW w AS (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL});
"""
con.execute(FEAT_SQL); open("sql/features.sql", "w").write(FEAT_SQL)
feat = con.execute("SELECT * FROM feat").df(); print(feat.shape); display(feat.head(3))
feat = feat.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

(109141, 26)


,plant,ts,cf,GHI,DNI,cloud,temp,hr,dow,mon,doy,y_lag1,y_lag2,y_lag3,y_lag24,y_lag48,y_lag168,y_ma_season,y_sd_season,y_diff1,GHI_lag_season,DNI_lag_season,cloud_lag_season,temp_lag_season,y_h1,y_h6
0,Actual_37.75_-122.05_2006_UPV_,2006-11-08 11:00:00,0.409024,180.0,0.0,3.0,17.7,11,3,11,312,0.482433,0.046783,0.303809,0.559357,0.469305,0.773299,0.364009,0.282170,-0.073410,450.0,495.0,7.0,23.7,0.579512,0.068158
1,Actual_37.75_-122.05_2006_UPV_,2006-11-08 12:00:00,0.579512,342.0,56.0,3.0,17.7,12,3,11,312,0.409024,0.482433,0.046783,0.640717,0.252589,0.663462,0.361458,0.279828,0.170488,274.0,207.0,8.0,23.3,0.767936,0.083247
2,Actual_37.75_-122.05_2006_UPV_,2006-11-08 13:00:00,0.767936,558.0,937.0,0.0,17.1,13,3,11,312,0.579512,0.409024,0.482433,0.529956,0.251775,0.693787,0.371374,0.290087,0.188425,232.0,635.0,1.0,21.7,0.705067,0.098336


In [36]:
# Step 3e: leakage check: for every row, the largest lag feature must come from a timestamp strictly before the target timestamp (by construction) and
# the target of horizon h must equal the raw target h steps later within the same unit
g0 = feat[feat[UNIT_COL] == feat[UNIT_COL].iloc[0]].sort_values(TIME_COL).reset_index(drop=True)
h = HORIZONS[0]; ok = np.allclose(g0[f"y_h{h}"].iloc[:-h].values, g0[TARGET_COL].iloc[h:].values, equal_nan=True)
print("horizon target aligned with raw target shifted by h:", ok)
assert ok, "target alignment broken: check FREQ and the resample grid"

horizon target aligned with raw target shifted by h: True


In [37]:
# Step 3f: RQ1 figures 2 and 3 for the paper: event share per unit and target by covariate decile (from the SQL outputs)
q2 = pd.read_csv("report/table_q2.csv"); q3 = pd.read_csv("report/table_q3.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
b = axes[0].bar(range(len(q2)), q2.event_share, color=TEAL); axes[0].set_xlabel("units"); axes[0].set_ylabel("event share"); style(axes[0])
axes[1].plot(q3.decile, q3.mean_target, marker="o", color=ACC); axes[1].set_xlabel("covariate decile"); axes[1].set_ylabel("mean " + TARGET_COL); style(axes[1])
savefig(fig, "fig_rq1_sql")

saved report/fig_rq1_sql.png


## Testing, handoff, and the Audit Log

**Test cases** must all pass before handoff. **Manifest** records row count, columns, time range, and the parquet's MD5 hash so Student B can confirm they received the correct file. Then fill in the **AI Audit Log** for prompts used in weeks 1 and 2 (at least 3 entries per week, with at least one entry catching the AI being wrong).

In [38]:
# Test cases for Notebook_C (all asserts must pass)
assert feat[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert feat[TARGET_COL].notna().all(), "target missing after cleaning"
assert set(HORIZONS) <= set(int(c[3:]) for c in feat.columns if c.startswith("y_h")), "horizon targets missing"
assert (feat.groupby(UNIT_COL)[TIME_COL].apply(lambda s: s.is_monotonic_increasing)).all(), "time not sorted within unit"
assert len(feat) >= 30000, f"only {len(feat)} rows: collect more data (Step 1d) or lower FREQ"
print("Tests C: OK")

Tests C: OK


In [39]:
# Handover: parquet + manifest with md5
feat.to_parquet("data/processed/feat.parquet", index=False)
feat.to_csv("data/processed/feat.csv", index=False)                    # CSV copy of the handover table (parquet is the file of record)
md5 = hashlib.md5(open("data/processed/feat.parquet", "rb").read()).hexdigest()
manifest = {"group": GROUP, "topic": TOPIC, "rows": len(feat), "cols": list(feat.columns), "units": int(feat[UNIT_COL].nunique()), "start": str(feat[TIME_COL].min()), "end": str(feat[TIME_COL].max()),
            "freq": FREQ, "horizons": HORIZONS, "season": SEASON, "unit_col": UNIT_COL, "time_col": TIME_COL, "target_col": TARGET_COL, "exog_cols": [c for c in EXOG_COLS if c in feat.columns], "test_start": TEST_START, "md5": md5}
json.dump(manifest, open("data/processed/manifest.json", "w"), indent=2); print(json.dumps(manifest, indent=2)[:600])

{
  "group": "Group 7",
  "topic": "Cross-plant Transfer and Conformal Uncertainty for Hourly Solar Power Forecasting on NREL Integration Data",
  "rows": 109141,
  "cols": [
    "plant",
    "ts",
    "cf",
    "GHI",
    "DNI",
    "cloud",
    "temp",
    "hr",
    "dow",
    "mon",
    "doy",
    "y_lag1",
    "y_lag2",
    "y_lag3",
    "y_lag24",
    "y_lag48",
    "y_lag168",
    "y_ma_season",
    "y_sd_season",
    "y_diff1",
    "GHI_lag_season",
    "DNI_lag_season",
    "cloud_lag_season",
    "temp_lag_season",
    "y_h1",
    "y_h6"
  ],
  "units": 20,
  "start": "2006-01-01 07:0


In [40]:
# Record the prompts you used this week (edit the examples; keep only real prompts)
audit("Step 1d", f"Topic {TOPIC}: suggest a secondary source joinable on {TIME_COL}", "Claude", "3 sources suggested", "opened URLs, checked license", "chose source ..., excluded source ... because ...")
audit("Step 2c", f"Column {TARGET_COL} is missing {round(float(gap.mean()), 3)} in streaks; interpolate or drop?", "Claude", "recommended interpolate(limit=3)", "checked that no future values were used", "applied limit=3")
print(pd.read_csv(AUDIT_PATH).tail(3))

audit entry saved: Step 1d
audit entry saved: Step 2c
         date    group     step                                             prompt    tool                         ai_output  \
1  2026-09-18  Group 7  Step 2c  Column cf is missing 0.501 in streaks; interpo...  Claude  recommended interpolate(limit=3)   
2  2026-09-18  Group 7  Step 1d  Topic Cross-plant Transfer and Conformal Uncer...  Claude               3 sources suggested   
3  2026-09-18  Group 7  Step 2c  Column cf is missing 0.501 in streaks; interpo...  Claude  recommended interpolate(limit=3)   

                              verified_how                                           decision  hallucination  
1  checked that no future values were used                                    applied limit=3              0  
2             opened URLs, checked license  chose source ..., excluded source ... because ...              0  
3  checked that no future values were used                                    applied limit=3       

## Extension exercises for Student A (do at least 3 of 5, record results in report/exercises_C.md)

1. **A third source**: add one more secondary source (holiday calendar, prices, events) using the same `load_table` function and `merge_asof`; report the row match rate and correlation with the target.
2. **Frequency comparison**: aggregate the target to a coarser frequency (e.g. daily) and compare seasonal strength between the two frequencies; conclude which frequency suits your topic's question.
3. **Two additional SQL queries**: one query using `QUALIFY` to get the top 5 units per month, one `WITH` query computing the event share by season; explain the results in two sentences.
4. **Seasonal decomposition**: use `statsmodels.tsa.seasonal.STL` on one unit; plot trend, seasonal, resid; comment on whether the residual still shows structure.
5. **Deliberate leakage check**: create one deliberately wrong feature (using `LEAD`) and have Student B test-run it to see the MAE drop unusually; record the lesson learned in the Discussion.

In [41]:
# Exercise 4 starter: STL decomposition for one unit (period = SEASON)
from statsmodels.tsa.seasonal import STL
u0 = df[UNIT_COL].unique()[0]; s = df[df[UNIT_COL] == u0].set_index(TIME_COL)[TARGET_COL].asfreq(FREQ).interpolate(limit=3).dropna()
if len(s) > 3 * SEASON:
    stl = STL(s, period=SEASON, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for ax, comp, lab in zip(axes, [stl.trend, stl.seasonal, stl.resid], ["trend", "seasonal", "residual"]): ax.plot(comp, color=TEAL, lw=0.8); ax.set_ylabel(lab); style(ax)
    savefig(fig, "fig_exercise_stl"); print("residual share of variance:", round(float(stl.resid.var() / s.var()), 3))

saved report/fig_exercise_stl.png
residual share of variance: 0.939


In [46]:
# Exercise 2 starter: seasonal strength at a coarser frequency (daily) versus the working frequency
daily = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("D").mean().reset_index()
strength = lambda frame, key: round(frame.groupby(key)[TARGET_COL].mean().std() / frame[TARGET_COL].std(), 3)
print("weekday strength: working freq", strength(df.assign(k=df[TIME_COL].dt.dayofweek), "k"), "| daily", strength(daily.assign(k=daily[TIME_COL].dt.dayofweek), "k"))
print("week strength: working freq", strength(df.assign(k=df[TIME_COL].dt.isocalendar().week), "k"),
      "| daily", strength(daily.assign(k=daily[TIME_COL].dt.isocalendar().week), "k"))
print("month strength: working freq", strength(df.assign(k=df[TIME_COL].dt.month), "k"), "| daily", strength(daily.assign(k=daily[TIME_COL].dt.month), "k"))

weekday strength: working freq 0.032 | daily 0.088
week strength: working freq 0.134 | daily 0.371
month strength: working freq 0.105 | daily 0.291


In [52]:
# Exercise 3 starter: QUALIFY and CTE examples (edit and add your own two queries to sql/queries.sql)
ex_sql = f"""
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS m, RANK() OVER (PARTITION BY month({TIME_COL}) ORDER BY AVG({TARGET_COL}) DESC) AS rk FROM clean GROUP BY 1, 2 QUALIFY rk <= 5 ORDER BY mon, rk;
"""
ex_sql2 = f"""
WITH seasons AS (
  SELECT *, CASE WHEN month({TIME_COL}) IN (12,1,2) THEN 'winter'
                 WHEN month({TIME_COL}) IN (3,4,5) THEN 'spring'
                 WHEN month({TIME_COL}) IN (6,7,8) THEN 'summer'
                 ELSE 'fall' END AS season
  FROM clean
),
thr AS (SELECT quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS q FROM clean)
SELECT season, AVG(({TARGET_COL} > q)::INT) AS event_share FROM seasons, thr GROUP BY 1 ORDER BY 2 DESC;
"""
display(con.execute(ex_sql).df())
full = con.execute(ex_sql).df()
consistency = full.groupby('plant').size().sort_values(ascending=False).rename('months_in_top5')
print(consistency)
display(con.execute(ex_sql2).df())

,plant,mon,m,rk
0,Actual_33.75_-116.15_2006_UPV_,1,0.461538,1
1,Actual_34.95_-118.25_2006_UPV_,1,0.443209,2
2,Actual_35.05_-118.15_2006_UPV_,1,0.432045,3
3,Actual_32.75_-115.45_2006_UPV_,1,0.426823,4
4,Actual_33.25_-114.85_2006_UPV_,1,0.391517,5
5,Actual_33.75_-116.15_2006_UPV_,2,0.520419,1
6,Actual_35.05_-118.15_2006_UPV_,2,0.478436,2
7,Actual_34.95_-118.25_2006_UPV_,2,0.460618,3
8,Actual_32.75_-115.45_2006_UPV_,2,0.428650,4
9,Actual_33.75_-116.25_2006_DPV_,2,0.400546,5


plant
Actual_32.75_-115.45_2006_UPV_    12
Actual_33.75_-116.15_2006_UPV_    12
Actual_34.95_-118.25_2006_UPV_    12
Actual_35.05_-118.15_2006_UPV_    12
Actual_37.75_-122.05_2006_UPV_     6
Actual_33.25_-114.85_2006_UPV_     4
Actual_33.75_-116.25_2006_DPV_     2
Name: months_in_top5, dtype: int64


,season,event_share
0,spring,0.149641
1,summer,0.102992
2,fall,0.085057
3,winter,0.050686


In [55]:
# Generate report/exercises_C.md — fill in your own comments/observations below, then run this cell
import os
os.makedirs('report', exist_ok=True)

exercises_md = """# Extension Exercises — Notebook C

## Exercise 4: STL Decomposition (trend / seasonal / residual)

**Residual share of variance:** 0.939
The residual still shows structure and doesn't look like random noises as it stays quite constant throughout the year with March fluctuating the most, and July fluctuating the least

---

## Exercise 2: Seasonal Strength at Different Frequencies

**Weekday strength, hourly:** 0.032
**Weekday strength, daily:** 0.088
**Week strength, hourly:** 0.134
**Week strength, daily:** 0.371
**Month strength, hourly:** 0.105
**Month strength, daily:** 0.291

**Conclusion — which frequency suits this topic's question:**
Daily-averaged data consistently shows stronger seasonal signal than raw hourly data across all three groupings (roughly 2.5–3x stronger), since daily averaging removes day/night noise that dilutes hourly comparisons. Among the three groupings, week captures the strongest signal (0.371 at daily frequency), stronger than month (0.291), suggesting real seasonal transitions happen at a finer timescale than a full month and get partially smoothed away by month-level aggregation. Weekday (day of the week) is by far the weakest (0.088), consistent with having no physical relationship to solar output. This suggests that if seasonality needs to be captured as a model feature, week is a better choice than month, and weekday should likely be excluded as a feature entirely

---

## Exercise 3: Extra SQL Queries (QUALIFY + CTE)

**Query 1 (QUALIFY — top 5 plants per month):**
The query found 4 plants that appeared in the top 5 in all 12 months, implying some plants perform better than other (may be due to location)

**Query 2 (WITH/CTE — event share by season):**
This query found that as the season progresses (spring -> summer -> fall -> winter), the event_share goes down (from ~0.15 to barely 0.05), so most of the high performances are at the beginning of the year.
This also reveals Summer's event_share at roughly 0.10, which is significantly lower than Spring, this could mean the extreme heat of Summer causes the performances to be worse.

---

## Summary
The STL residual has substantial structure (large variance share), suggesting a fixed trend+seasonal decomposition doesn't fully capture short-term weather variability. Seasonal strength is consistently ~2.75x higher when computed on daily-averaged data than hourly data across weekday, week, and month groupings, with week showing the strongest signal (0.371) meaning week could be a valuable additional feature for capturing seasonal position. Four plants ranked in the top 5 by mean cf in all 12 months, indicating some plants consistently outperform others rather than varying by seasonal luck alone. Event share is highest in spring (15.0%) and lowest in winter (5.1%), with summer notably lower than spring (10.3%), a pattern worth exploring further, possibly linked to reduced panel efficiency at high summer temperatures, though this is just a hypothesis.

"""

with open('report/exercises_C.md', 'w', encoding='utf-8') as f:
    f.write(exercises_md)

print("Created report/exercises_C.md — open it and fill in the <fill in> placeholders with your actual results and comments.")

Created report/exercises_C.md — open it and fill in the <fill in> placeholders with your actual results and comments.


### Self-assessment for Notebook_C (fill in before sending to the instructor)

| Criterion | Self-score (0-2) | Evidence |
|---|---|---|
| Sources and licenses are clear | 1.5 | report/data_sources.csv |
| Two-source comparison with numbers | 2 | table_source_comparison.csv |
| Cleaning has a log | 2 | table_cleaning_log.csv |
| EDA has observations with numbers | 2 | eda_notes.md |
| SQL has no leakage, has been checked | 2 | features.sql, leakage-check cell |
| Extension exercises | 2 | exercises_C.md |
| AI Audit Log | 0 | ai_audit_log.csv (inconsistent/outdated format, AI was only used to help fix bugs and understand, and help with exercises) |

## Common errors and how to fix them

| Error | Cause | Fix |
|---|---|---|
| `KeyError: 'ts'` | the column name in the file differs from CONFIG | print `df.columns`, fix `TIME_COL` |
| Future leakage (suspiciously good score) | used `shift(-k)` or a whole-series mean when building a feature | only use `LAG` and `PRECEDING` windows; check that a feature's `feat.ts` is always earlier than the target |
| Duplicate rows by unit and time | the source has two records at the same timestamp | `drop_duplicates([UNIT_COL, TIME_COL])` then record it in the cleaning log |
| Time grid missing steps | didn't `resample(FREQ)` | resample and count missing steps per unit |
| Merging the secondary source loses half the rows | timezone or frequency mismatch | convert both to UTC or a consistent local time, use `merge_asof` with `tolerance` |
| DuckDB `Binder Error` | column name is a keyword (`do`, `at`, `year`) or clashes with a table name | rename the column, use double quotes |
| `MemoryError` | loading an entire large file with pandas | use `duckdb.read_csv_auto` and filter early |
| Chronos won't install | missing `pip install chronos-forecasting` | skip the foundation-model cell, note it as a limitation |

## Notebook_C checklist before handoff (tick each line)

- [x] Three CSV files: `data/raw/primary_snapshot.csv`, `data/processed/clean.csv`, `data/processed/feat.csv` open in Excel, row counts match the manifest
- [x] `report/data_sources.csv` has both sources with URL, license, row count, time range
- [x] `report/table_source_comparison.csv` and one sentence on how consistent the two sources are
- [x] `report/table_cleaning_log.csv` explains every dropped row
- [x] 9 EDA figures with captions stating numbers; `report/eda_notes.md` has 5 observations with numbers
- [x] `sql/queries.sql` 4 RQ1 queries run successfully; `report/table_q1..q4.csv`; `report/table_rq1_tests.csv` has p-values
- [x] `sql/features.sql` uses no LEAD outside the target; the leakage-check cell passes
- [x] Test cases pass; `feat.parquet` and `manifest.json` sent to Student B
- [ ] AI Audit Log has at least 6 entries for weeks 1 and 2, with at least one `hallucination=True` entry

## Template for writing the Data and RQ1 sections of the paper (fill in numbers from the tables)

**Data.** We use {PRIMARY_SOURCE} ({license}, {rows} rows, {units} units, {start} to {end}, frequency {FREQ}) merged with {SECOND_SOURCE} ({share}% of rows have covariates). Cleaning removed {removed} rows (Table cleaning log); the two sources agree on {metric} (Table source comparison).

**RQ1.** The target varies most with {calendar or covariate}: seasonal strength {value} by {period} versus {value} by {period} (Figure EDA 4); units differ by a factor of {ratio} (Figure EDA 5); the covariate {name} leads the target by {lag} steps with correlation {rho} (Figure EDA 7); autocorrelation at the seasonal lag is {acf}, so the seasonal naive baseline is expected to be strong (Figure EDA 8). Kruskal-Wallis across units gives p = {p} (Table RQ1 tests).